In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from transformers import BertTokenizer
# import torch as pt
# import re
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from imblearn.over_sampling import RandomOverSampler
# from tqdm import tqdm
# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
# from sklearn.model_selection import train_test_split
# from gensim.models import Word2Vec

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# 1. Load GloVe Embeddings
def load_glove_embeddings(glove_file_path):
    print(f"Loading GloVe embeddings from {glove_file_path}...")
    embeddings_index = {}
    try:
        with open(glove_file_path, encoding='utf-8') as f:
            for line in f:
                values = line.split()
                word = values[0]
                # Handle cases where a line might be malformed
                try:
                    coefs = np.asarray(values[1:], dtype='float32')
                    embeddings_index[word] = coefs
                except ValueError:
                    continue
        print(f"Found {len(embeddings_index)} word vectors.")
        return embeddings_index
    except FileNotFoundError:
        print(f"Error: File not found at {glove_file_path}")
        print("Please ensure 'glove.6B.100d.txt' is in the correct directory.")
        return {}

# 2. Generate Averaged Embeddings (Pure Python/Numpy)
def get_averaged_glove_features(texts, embeddings_index, embedding_dim=100):
    """
    Converts a list of text strings into a list of averaged GloVe vectors.
    """
    features = []
    for text in tqdm(texts, desc="Vectorizing texts"):
        if not isinstance(text, str):
            text = ""
            
        # Simple tokenization: lowercase and split by whitespace
        words = text.lower().split()
        
        # Filter for words present in GloVe
        valid_vectors = [embeddings_index[w] for w in words if w in embeddings_index]
        
        if valid_vectors:
            # Average the vectors
            avg_vector = np.mean(valid_vectors, axis=0)
        else:
            # If no words found (or empty text), return zero vector
            avg_vector = np.zeros(embedding_dim)
            
        features.append(avg_vector)
        
    return np.array(features)

In [ ]:
glove_path = 'dolma_300_2024_1.2M.100_combined.txt' 

embeddings_index = load_glove_embeddings(glove_path)

if embeddings_index:
    # Load Data
    print("Loading dataset...")
    df = pd.read_csv("processed_datasets/WELFake_Dataset_processed.csv")
    
    # Ensure text column is string
    df['combined_text'] = df['combined_text'].astype(str)
    
    texts = df['combined_text'].tolist()
    labels = df['label'].values

    # FIX: Determine embedding dimension from the loaded data automatically
    # This prevents mismatch between loaded vectors and the zero-vector fallback
    if len(embeddings_index) > 0:
        sample_key = next(iter(embeddings_index))
        embedding_dim = len(embeddings_index[sample_key])
        print(f"Detected embedding dimension: {embedding_dim}")
    else:
        embedding_dim = 100 # Fallback default

    # Generate Features
    X_glove = get_averaged_glove_features(texts, embeddings_index, embedding_dim)

    # Create DataFrame
    glove_feature_cols = [f'glove_{i}' for i in range(embedding_dim)]
    glove_df = pd.DataFrame(X_glove, columns=glove_feature_cols)
    glove_df['label'] = labels

    # Save
    output_path = 'preprocessed_datasets/glove_features_averaged.csv'
    glove_df.to_csv(output_path, index=False)
    print(f"Averaged GloVe features saved to {output_path}")

Loading GloVe embeddings from dolma_300_2024_1.2M.100_combined.txt...
Found 1200001 word vectors.
Loading dataset...
Detected embedding dimension: 300


Vectorizing texts: 100%|██████████| 72124/72124 [00:47<00:00, 1530.31it/s]


Averaged GloVe features saved to preprocessed_datasets/glove_features_averaged.csv


In [ ]:
df = pd.read_csv("preprocessed_datasets/glove_features_averaged.csv")

df.head()

,glove_0,glove_1,glove_2,glove_3,glove_4,glove_5,glove_6,glove_7,glove_8,glove_9,...,glove_291,glove_292,glove_293,glove_294,glove_295,glove_296,glove_297,glove_298,glove_299,label
0,-0.087969,-0.065743,-0.197766,-0.025432,0.044824,-0.132333,-0.161037,-0.006665,0.918580,-0.023367,...,0.045207,0.174155,-0.022262,-0.112820,0.003164,0.193079,0.372457,-0.013245,0.011839,1
1,-0.097187,-0.049914,-0.311329,-0.189515,0.029972,-0.138930,-0.164260,0.009709,1.180396,0.048713,...,-0.040974,0.188815,0.048454,-0.164614,-0.097104,0.213931,0.277798,0.109681,0.015956,1
2,-0.159360,-0.058271,-0.151604,-0.023289,0.012202,-0.233306,-0.113569,-0.003830,0.783874,-0.082330,...,0.067064,0.083402,-0.020227,-0.103164,0.012808,0.149834,0.359297,0.108930,-0.006429,1
3,-0.126454,-0.056505,-0.172090,-0.031410,-0.004020,-0.158922,-0.143521,0.000892,0.906343,-0.011381,...,0.041894,0.178555,0.003800,-0.102395,0.000989,0.198056,0.344069,-0.030789,0.027161,0
4,-0.072542,-0.118940,-0.174961,0.031754,0.065772,-0.177495,-0.108461,0.074498,0.619597,-0.005935,...,-0.010755,0.178652,0.021008,-0.082601,0.052799,0.152559,0.316286,-0.005364,0.000829,1


In [ ]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [ ]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())
    
    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)
    
    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [ ]:
def train_ml_models(xtrain, ytrain, xtest, ytest):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))
    
    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression()
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Support Vector Classifier \n")
    svc_model = SVC()
    svc_model = svc_model.fit(xtrain, ytrain)

    ypred = svc_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

[0 1 1 ... 0 1 0]
[1 1 1 ... 0 1 0]
              precision    recall  f1-score   support

           0       0.81      0.91      0.86      6982
           1       0.90      0.80      0.85      7443

    accuracy                           0.85     14425
   macro avg       0.86      0.85      0.85     14425
weighted avg       0.86      0.85      0.85     14425

